# Model 11

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"     # agama
os.environ["MKL_NUM_THREADS"] = "1"     # numpy, scipy
os.environ["OPENBLAS_NUM_THREADS"] = "1"    # numpy
os.environ["NUMEXPR_NUM_THREADS"] = "1"     # pandas

import agama
import torch 
import numpy as np
from scipy import integrate
from scipy.stats import wasserstein_distance
from astropy import units as u

from sbi.utils import BoxUniform
from sbi.inference import SNLE, simulate_for_sbi, prepare_for_sbi
from sbi.utils import likelihood_nn

from sklearn.metrics import mean_squared_error, r2_score

import pandas as pd
import pickle
import matplotlib.pyplot as plt
from galaxy_generation import generate_galaxy_multiple
from prior_generation import generate_prior
from standardization import get_standard, standardize
import corner

torch.set_num_threads(1)


In [2]:
# set agama unit to be in Msun, kpc, km/s
agama.setUnits(mass=1 * u.Msun, length=1*u.kpc, velocity=1 * u.km /u.s)
agama.setRandomSeed(13)
torch.manual_seed(13)
np.random.seed(13)


## Generate Galaxy

In [13]:
num_galaxies = 1
prior = generate_prior()
# theta = prior.sample((num_galaxies,))
theta = torch.tensor([[7, 0, 0, 0.2]])
# n_stars = np.random.poisson(1000, size=num_galaxies)
n_stars = [1000]

theta = torch.repeat_interleave(theta, torch.tensor(n_stars), dim=0)
x = generate_galaxy_multiple(theta, n_stars, 1, d=3)

In [14]:
pd.DataFrame(x).to_csv("mass_density_x_core_model_14_s1000.csv", index=None, header=None)
# pd.DataFrame(theta).to_csv("train_theta_model_13_1.csv", index=None, header=None)

### Reuse same datasets model 11

In [3]:
data = "mass_density_x_cusp_model_11"
df = np.array(pd.read_csv(f"{data}.csv", header=None))

out = np.zeros((df.shape[0],2))
out[:,0] = np.sqrt(df[:,0] ** 2 + df[:,1] ** 2)
out[:,1] = df[:, -1]

pd.DataFrame(out).to_csv(f"{data}.csv", header=None, index=None)

## Generate galaxy for model 14

In [3]:
data = "x_o_cusp"
df = pd.read_csv(f"{data}.csv", header=None)
df = df.drop(columns=[2, 3, 4])
df.to_csv(f"{data}.csv", header=None, index=None)

## Contour simulations

In [ ]:
with open("inference_model_11.pkl", "rb") as file:
    inference = pickle.load(file)

cored, gamma = 0

In [ ]:
galaxy = np.expand_dims(np.array([7, 0, 0, 0.2]), axis=0).astype(np.float32)
df = inference._neural_net.sample(100_000, context=galaxy).squeeze().detach().numpy()
pd.DataFrame(df).to_csv("contour_samples_model_11_core.csv", header=None, index=None)

cuspy, gamma =1

In [ ]:
galaxy = np.expand_dims(np.array([7, 0, 1, 0.2]), axis=0).astype(np.float32)
df = inference._neural_net.sample(100_000, context=galaxy).squeeze().detach().numpy()
pd.DataFrame(df).to_csv("contour_samples_model_11_cusp.csv", header=None, index=None)


### Optuna

In [2]:
with open("tune_model_11_all.pkl", "rb") as file:
    o = pickle.load(file)

In [4]:
o.best_trial

FrozenTrial(number=5, state=<TrialState.COMPLETE: 1>, values=[0.34845529465945724], datetime_start=datetime.datetime(2026, 6, 16, 22, 56, 24, 658263), datetime_complete=datetime.datetime(2026, 6, 16, 23, 19, 5, 218642), params={'learning_rate': 0.0010413849605126644, 'training_batch_size': 1024}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'learning_rate': FloatDistribution(high=0.01, log=True, low=0.0001, step=None), 'training_batch_size': CategoricalDistribution(choices=(256, 512, 1024, 2048, 4096))}, trial_id=5, value=None)

### Sequential x_o

In [5]:
df = np.array(pd.read_csv("mass_density_x_cusp_model_12.csv", header=None))

In [6]:
l = np.zeros((df.shape[0], 2))
l[:, 0] = np.sqrt(df[:,0] ** 2 + df[:,1] ** 2)
l[:, 1] = df[:, -1]

In [7]:
pd.DataFrame(l).to_csv("mass_density_x_cusp_model_12.csv", header=None, index=None)

In [2]:
with open("samples_model_11_test.pkl", "rb") as file:
    i = pickle.load(file)

In [8]:
np.min(i[0], axis=0)

array([ 5.0014057, -0.9997709, -0.9974579,  0.2024671], dtype=float32)

In [6]:
np.median(i[0], axis=0)

array([ 6.208494  , -0.2037257 ,  0.6861227 ,  0.55519307], dtype=float32)

In [6]:
test_theta_raw = np.array(pd.read_csv("train_theta.csv", header=None))
test_x_raw = np.array(pd.read_csv("train_x.csv", header=None))
l = standardize(test_theta_raw, test_x_raw)

In [9]:
l[1]

(tensor([[-0.2603,  0.0326],
         [ 0.2444,  0.0058],
         [-0.2681,  0.0385],
         ...,
         [-0.2371, -0.0222],
         [ 0.0705, -0.0791],
         [-0.0281, -0.0946]]),
 array([1.32582864, 0.00187228]),
 array([ 4.46006741, 13.61989592]))

In [3]:
with open("inference_model_11.pkl", "rb") as file:
    inference = pickle.load(file)